# Computational Theory: SHA-256 Implementation

## Problem 1: Representing SHA-256 Data

Explain how you would represent:

32-bit words,
sequences of 32-bit words,
input messages,
512-bit message blocks,
the final 256-bit hash value,
big-endian integers.
For each representation, explain why it is appropriate and demonstrate your choices with small Python examples.

### 32-bit word
A 32-bit word is a value between 0 and 2^32 - 1 (0x00000000 to 0xFFFFFFFF). Python ints can hold larger values, so the mask `& 0xFFFFFFFF` keeps only the lowest 32 bits when simulating 32-bit overflow. Integers are convenient here because Python supports the bitwise operations SHA-256 uses.

In [2]:
# A 32-bit word constant
w = 0x6A09E667

# Simulating 32-bit addition with overflow truncation
a = 0xFFFFFFFF
b = 0x00000005
result = (a + b) & 0xFFFFFFFF  # Keeps only the lowest 32 bits

print(f"Hex: {hex(result)}")  # Hex: 0x4

Hex: 0x4


### Sequences of 32-bit Words
A series of 32-bit values can be represented by list of ints in python. Why it's appropriate: Lists support fast indexed lookups (words[i]), slice access, and in-place updates, making them intuitive for algorithmic round loops.


In [3]:
# Initial state of 8 working variables (each a 32-bit word)
H = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19
]

# Accessing word 0
print(f"First word: {hex(H[0])}")

First word: 0x6a09e667


### Input Messages
The raw data before padding and hashing (text, files, or binary streams). Use bytes for this, array of bytes. Becasue hashing operates purely on raw binary bytes. So you convert strings into bytes.

In [ ]:
# Converting user text to raw bytes using UTF-8
message_str = "hi, ivan"
message_bytes = message_str.encode("utf-8") 
# OR 
message_bytes_2 = b"hi, ivan"

print(message_bytes)        # b'hi, ivan'
print(list(message_bytes))  # [104, 105, 44, 32, 105, 118, 97, 110]
print(message_bytes_2)        # b'hi, ivan'
print(list(message_bytes_2))  # [104, 105, 44, 32, 105, 118, 97, 110]

b'hi, ivan'
[104, 105, 44, 32, 105, 118, 97, 110]
b'hi, ivan'
[104, 105, 44, 32, 105, 118, 97, 110]


### 512-bit Message Blocks
Hashing algorithms process padded input in chunks of 512 bits (which equals 64 bytes, or 16 thirty-two-bit words). Chosen representation: A bytes object of length 64, or a list of 16 ints unpacked from it. Why it works: SHA-256 takes 64-byte chunks directly from the padded message and immediately slices or unpacks them into 16 individual 32-bit words for processing.

In [6]:
import struct

# A 64-byte (512-bit) chunk
block_bytes = b"A" * 64
print(f"Block byte length: {len(block_bytes)}")  # 64 bytes = 512 bits

# Unpack 64 bytes into sixteen 32-bit big-endian words
# '>16I' means: '>' (big-endian), '16' (count), 'I' (unsigned 32-bit int)
words = list(struct.unpack(">16I", block_bytes))
print(f"Number of words: {len(words)}")  # 16

Block byte length: 64
Number of words: 16


### Final 256-bit Hash Value

The completed SHA-256 digest is 256 bits (32 bytes, or eight 32-bit words). It can be represented as bytes for binary use or as a 64-character hexadecimal string for display.

In [7]:
# Known SHA-256 digest for the message b"abc", shown here as eight words.
H_final = [
    0xBA7816BF, 0x8F01CFEA, 0x414140DE, 0x5DAE2223,
    0xB00361A3, 0x96177A9C, 0xB410FF61, 0xF20015AD
]

# Format as 64-character hex string (each word is 8 hex chars, padded with zeros)
hex_digest = "".join(f"{word:08x}" for word in H_final)
print(f"Hex digest ({len(hex_digest)} chars): {hex_digest}")

# Or convert to 32 raw bytes
import struct
raw_digest = struct.pack(">8I", *H_final)
print(f"Raw digest length: {len(raw_digest)} bytes")  # 32 bytes = 256 bits

Hex digest (64 chars): ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
Raw digest length: 32 bytes


### Big-Endian Integers

Big-endian stores the most significant byte first. For example, `b"\x12\x34\x56\x78"` represents the integer `0x12345678` when interpreted as big-endian. Python's `int.from_bytes(..., byteorder="big")`, `int.to_bytes(..., byteorder="big")`, and `struct`'s `>` prefix make this byte order explicit.

In [ ]:
# 1. Converting 4 bytes to a 32-bit big-endian integer
byte_data = b"\x12\x34\x56\x78"

# 0x12 is the highest byte, 0x78 is the lowest
number = int.from_bytes(byte_data, byteorder="big")
print(f"Integer to hex: {hex(number)}")  # 0x12345678

# 2. Converting an integer back to 4 big-endian bytes
packed_bytes = number.to_bytes(4, byteorder="big")
print(f"Back to bytes: {packed_bytes}")   # b'\x124Vx' (hex 0x12, 0x34, 0x56, 0x78)

## Problem 2: SHA-256 Bitwise Operations



$\text{Ch}(x, y, z)$ (Choose) selects each output bit from $y$ when the corresponding bit of $x$ is 1, and from $z$ when it is 0. SHA-256 uses this bitwise function in each compression round.

In [17]:
import numpy as np

def Ch(x: np.uint32, y: np.uint32, z: np.uint32) -> np.uint32:
    """
    Computes the SHA-256 Choose function: Ch(x, y, z) = (x AND y) XOR (NOT x AND z).
    
    For each bit position:
      - If the bit in x is 1, the output bit is taken from y.
      - If the bit in x is 0, the output bit is taken from z.
      
    Parameters:
        x, y, z : np.uint32 or int
            32-bit unsigned integers.
            
    Returns:
        np.uint32: 32-bit result of the bitwise selection.
    """
    # Enforce 32-bit unsigned integer types
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)
    
    # Compute: (x AND y) XOR ((NOT x) AND z)
    # For example, x = 1, y = 1, z = 0;
    # 1. x & y = 1 because they're both 1 that's how AND works, if one of them is 0 the result is 0
    # 2. ~x & z = 0 because ~x = opposite of x (NOT) = 0 and z = 0 anyway so you get 0 
    # 3. now we get 1 ^ 0 = 1 because of XOR, it gives you 1 if bits are different
    return np.uint32((x & y) ^ (~x & z))

assert Ch(1,1,0) == 1
assert Ch(0,0,0) == 0
assert Ch(0b1010, 0b0001, 0b1111) == 0b0101

print("Ch test passed")

Ch test passed


$\text{Maj}(x, y, z)$ (Majority) returns 1 at each bit position when at least two of the corresponding bits in $x$, $y$, and $z$ are 1. SHA-256 uses this bitwise function in each compression round.

In [15]:
def Maj(x: np.uint32, y: np.uint32, z: np.uint32) -> np.uint32:
    """
    Computes the SHA-256 Majority function: 
    Maj(x, y, z) = (x AND y) XOR (x AND z) XOR (y AND z).
    
    For each bit position, returns 1 if at least two of the three inputs have a 1,
    otherwise returns 0.
    
    Parameters:
        x, y, z : np.uint32 or int
            32-bit unsigned integers.
            
    Returns:
        np.uint32: 32-bit result of the majority vote.
    """
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)
    
    return np.uint32((x & y) ^ (x & z) ^ (y & z))

assert Maj(0,0,0) == 0
assert Maj(1,1,1) == 1
assert Maj(1,1,0) == 1
assert Maj(0,0,1) == 0
print("Maj tests passed")


Maj tests passed


The sigma functions each combine three rotated or shifted versions of one 32-bit word using XOR. Rotations move bits without discarding them; right shifts discard bits. The uppercase functions ($\Sigma_0$ and $\Sigma_1$) are used in the compression rounds, while the lowercase functions ($\sigma_0$ and $\sigma_1$) help expand the message schedule.

In [18]:
import numpy as np

#Helper code for functions, rotate and shift 

def rotr(x: np.uint32, n: int) -> np.uint32:
    """Rotates a 32-bit integer x to the right by n positions."""
    x = np.uint32(x)
    n = np.uint32(n)
    return np.uint32((x >> n) | (x << (np.uint32(32) - n)))

def shr(x: np.uint32, n: int) -> np.uint32:
    """Logical right shift of a 32-bit integer x by n positions."""
    return np.uint32(np.uint32(x) >> np.uint32(n))

Each function combines three transformed copies of its input with XOR. The uppercase functions use rotations only; the lowercase functions also use a right shift.

In [ ]:
#Big sigmas use 3 rotation operations but different number of shifting
def Sigma0(x: np.uint32) -> np.uint32:
    """
    SHA-256 Uppercase Sigma0:
    Sigma0(x) = ROTR^2(x) XOR ROTR^13(x) XOR ROTR^22(x)
    """
    x = np.uint32(x)
    return np.uint32(rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22))

def Sigma1(x: np.uint32) -> np.uint32:
    """
    SHA-256 Uppercase Sigma1:
    Sigma1(x) = ROTR^6(x) XOR ROTR^11(x) XOR ROTR^25(x)
    """
    x = np.uint32(x)
    return np.uint32(rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25))

#Small sigmas use 2 rotation operations and a shifting one with different number of shifts
def sigma0(x: np.uint32) -> np.uint32:
    """
    SHA-256 Lowercase sigma0:
    sigma0(x) = ROTR^7(x) XOR ROTR^18(x) XOR SHR^3(x)
    """
    x = np.uint32(x)
    return np.uint32(rotr(x, 7) ^ rotr(x, 18) ^ shr(x, 3))

def sigma1(x: np.uint32) -> np.uint32:
    """
    SHA-256 Lowercase sigma1:
    sigma1(x) = ROTR^17(x) XOR ROTR^19(x) XOR SHR^10(x)
    """
    x = np.uint32(x)
    return np.uint32(rotr(x, 17) ^ rotr(x, 19) ^ shr(x, 10))

x = 0x6a09e667

assert Sigma0(x) == 0xce20b47e, f"Got {hex(Sigma0(x))}"
assert Sigma1(x) == 0x55b65510, f"Got {hex(Sigma1(x))}"
assert sigma0(x) == 0xba0cf582, f"Got {hex(sigma0(x))}"
assert sigma1(x) == 0xcfe5da3c, f"Got {hex(sigma1(x))}"

print("All Sigma and sigma tests passed")

All Sigma and sigma tests passed


## Problem 3: Generating the SHA-256 Constants


### Background and Mathematical Formulation

The Secure Hash Standard (NIST FIPS PUB 180-4, Section 4.2.2) specifies a sequence of sixty-four 32-bit constant words, $K_0, K_1, \dots, K_{63}$, used during each round of the SHA-256 compression function. These constants break symmetries between rounds so that identical message blocks or states do not produce repetitive round behaviors.

To ensure these constants do not contain hidden backdoors, they are defined as "nothing-up-my-sleeve numbers" derived from the first sixty-four prime numbers ($2, 3, 5, \dots, 311$):

1. For each prime $p$, calculate its cube root: $\sqrt[3]{p} = p^{1/3}$.
2. Extract the fractional part: 
   $$\text{frac} = \sqrt[3]{p} - \lfloor \sqrt[3]{p} \rfloor$$
3. Take the first 32 bits of this binary fraction by multiplying by $2^{32}$ and truncating to an integer:
   $$K = \lfloor \text{frac} \times 2^{32} \rfloor$$

### 1. Prime Number Generation (`primes(n)`)

To generate the 64 constants, we first need a function `primes(n)` that outputs the first $n$ prime numbers. An integer $c \ge 2$ is prime if it has no divisors other than 1 and itself. A number $c$ is composite if and only if it has a prime factor $p \le \sqrt{c}$. Therefore, we can efficiently find primes by testing candidates against the primes already discovered up to $\sqrt{c}$.

**Reference:**
* National Institute of Standards and Technology (NIST). (2015). *Secure Hash Standard (SHS)* (FIPS PUB 180-4). Section 4.2.2 "SHA-224 and SHA-256 Constants".

In [1]:
def primes(n):
    """
    Generates the first n prime numbers using trial division.
    
    Parameters:
        n (int): The number of primes to generate.
        
    Returns:
        list[int]: A list containing the first n prime numbers.
    """
    if n <= 0:
        return []

    prime_list = []
    candidate = 2

    while len(prime_list) < n:
        is_prime = True
        for p in prime_list:
            if p * p > candidate:
                break
            if candidate % p == 0:
                is_prime = False
                break
        if is_prime:
            prime_list.append(candidate)
        candidate+=1
    
    return prime_list

assert primes(5) == [2, 3, 5, 7, 11]

## Problem 4: Padding and Parsing Messages


## Problem 5: The SHA-256 Compression Function


## Problem 6: Complete SHA-256
